<a href="https://colab.research.google.com/github/vkshadoww/114-2-Programing-Language/blob/main/Copy_of_HW4_PTT_GoogleSheet_RAG%E6%95%B4%E7%90%86%E7%89%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW4：Broadway.com → Google Sheet → RAG（整理版）

這份 notebook 保留完整流程：

1. 爬取 Broadway.com 文章
2. 寫入指定 Google Sheet
3. 從 Google Sheet 讀回資料
4. 建立 FAISS RAG 索引
5. 用 Gemini 根據 PTT 資料回答問題

主要修正：原本設定了 `SHEET_URL`，但實際用 `gc.open(WORKSHEET_NAME)` 開啟試算表，容易打開錯的 Spreadsheet。新版固定使用 `gc.open_by_url(SHEET_URL)`。


In [ ]:
# 安裝必要套件
!pip -q install gspread gspread-dataframe faiss-cpu sentence-transformers beautifulsoup4 requests google-generativeai


In [ ]:
import re
import time
import uuid
from datetime import datetime
from urllib.parse import urljoin

import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup

import faiss
from sentence_transformers import SentenceTransformer

import google.generativeai as genai
from google.colab import auth, userdata
import gspread
from google.auth import default
from gspread_dataframe import set_with_dataframe, get_as_dataframe


## 1. 基本設定

請確認 `SHEET_URL` 是你要寫入的 Google Sheet。  
`BROADWAY_WORKSHEET_NAME` 是存放原始文章的分頁。


In [ ]:
SHEET_URL = "https://docs.google.com/spreadsheets/d/1pA05rdBicdtbP1LZQIQrls8HxSqm74JgKmyFKAqLvzw/edit?usp=sharing"
BROADWAY_WORKSHEET_NAME = "broadway_shows"
TIMEZONE_NOTE = "Asia/Taipei"

BROADWAY_HEADER = [
    "show_id", "title", "url", "image_url", "description", "fetched_at"
]

BROADWAY_SHOWS_URL = "https://www.broadway.com/shows/tickets/"
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36" # Updated User-Agent


## 2. 連線 Google Sheet

這裡是最重要的修正：使用 `open_by_url(SHEET_URL)`，不要用 worksheet 名稱打開 spreadsheet。


In [ ]:
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 關鍵修正：直接用網址開啟指定 Google Sheet
sh = gc.open_by_url(SHEET_URL)
print(f"✅ 已開啟試算表：{sh.title}")
print(f"🔗 {SHEET_URL}")


In [ ]:
import time # Ensure time is imported at the top for sleep function

def ensure_worksheet(spreadsheet, title, header, rows=1000):
    """取得或建立 worksheet，並確保表頭正確。"""
    print(f"DEBUG: ensure_worksheet called for title: {title}")
    try:
        ws = spreadsheet.worksheet(title)
        print(f"DEBUG: Found existing worksheet: {title}")
    except gspread.WorksheetNotFound:
        print(f"DEBUG: Worksheet '{title}' not found. Creating new one.")
        ws = spreadsheet.add_worksheet(title=title, rows=str(rows), cols=str(len(header) + 5))
        ws.update([header])
        print(f"DEBUG: Created worksheet '{title}' and set header.")
        return ws

    values = ws.get_all_values()
    if not values:
        print(f"DEBUG: Worksheet '{title}' is empty. Setting header.")
        ws.update([header])
    else:
        # Strip whitespace from values[0] elements before comparison for robustness
        read_header = [str(col).strip() for col in values[0]]
        expected_header = [str(col).strip() for col in header] # Ensure header is also stripped just in case

        print(f"DEBUG: Worksheet '{title}' existing header: {read_header}")
        print(f"DEBUG: Expected header: {expected_header}")

        if read_header != expected_header:
            print(f"DEBUG: Header mismatch for '{title}'. Clearing and resetting header.")
            # 保留資料但重建欄位較危險，因此這裡直接清掉並重新建立正確表頭。
            # 若你要保留舊資料，請先備份 Google Sheet。
            ws.clear()
            ws.update([header])
        else:
            print(f"DEBUG: Header matches for '{title}'.")
    return ws


def read_sheet_df(ws, header):
    """從 worksheet 讀成 DataFrame，並清掉空列。"""
    df = get_as_dataframe(ws, evaluate_formulas=True, dtype=str).dropna(how="all")
    if df.empty:
        return pd.DataFrame(columns=header)
    df = df.loc[:, [c for c in df.columns if not str(c).startswith("Unnamed")]]
    for col in header:
        if col not in df.columns:
            df[col] = ""
    return df[header].fillna("")


def write_sheet_df(ws, df, header):
    """把 DataFrame 寫回 worksheet。"""
    df_out = df.copy()
    for col in header:
        if col not in df_out.columns:
            df_out[col] = ""
    df_out = df_out[header].infer_objects(copy=False).fillna("") # Add infer_objects to address FutureWarning

    # Google Sheet 寫入前統一轉字串，避免 Timestamp / NaN 型別問題
    for c in df_out.columns:
        df_out[c] = df_out[c].astype(str)

    ws.clear()
    set_with_dataframe(ws, df_out, include_index=False, include_column_header=True, resize=True)
    return len(df_out)


ws_broadway = ensure_worksheet(sh, BROADWAY_WORKSHEET_NAME, BROADWAY_HEADER)
print(f"✅ 已準備 worksheet：{ws_broadway.title}")

## 3. Broadway.com 爬蟲

這段只負責爬 Broadway.com，不碰 RAG。資料會先存在 `new_posts_df`。


In [ ]:
def now_iso():
    return datetime.now().isoformat(timespec="seconds")

In [ ]:
import time # Ensure time is imported at the top for sleep function
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup # Import BeautifulSoup

def make_show_id(url):
    return url.rstrip("/").split("/")[-1]

def parse_broadway_listing_card(card_container):
    # card_container is now expected to be <a class="swiper-carousel-card ...">
    # The link element is the container itself
    card_link_element = card_container

    title_tag = card_link_element.select_one("div.showlistpage__featured-shows__card__title")
    image_tag = card_link_element.select_one("img.showlistpage__featured-shows__card__img")

    # The URL comes from the 'href' of the card_link_element itself
    url = urljoin(BROADWAY_SHOWS_URL, card_link_element.get("href"))

    # Need to check if title and image tags exist within the card_link_element
    if not title_tag:
        print(f"DEBUG: parse_broadway_listing_card: No title tag found in card link: {card_link_element.prettify()[:200]}")
        return None
    if not image_tag:
        print(f"DEBUG: parse_broadway_listing_card: No image tag found in card link: {card_link_element.prettify()[:200]}")
        return None

    title = title_tag.get_text(strip=True)
    image_url = urljoin(BROADWAY_SHOWS_URL, image_tag.get("src"))
    show_id = make_show_id(url)

    return {
        "show_id": show_id,
        "title": title,
        "url": url,
        "image_url": image_url,
        "fetched_at": now_iso()
    }

def fetch_broadway_detail_description(show_url, delay=0.5):
    print(f"    Fetching detail page: {show_url} using Selenium")

    # Setup Selenium with headless Chrome - make sure to use the correct CHROMEDRIVER_PATH
    options = webdriver.ChromeOptions()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu') # Added for stability in headless environments
    options.add_argument('--window-size=1920,1080') # Added to specify window size
    options.add_argument('--disable-extensions') # Added to disable extensions
    options.add_argument('--disable-setuid-sandbox') # Added for more comprehensive sandbox disabling
    options.binary_location = '/usr/bin/google-chrome' # Explicitly specify Google Chrome binary

    CHROMEDRIVER_PATH = '/root/.cache/selenium/chromedriver/linux64/149.0.7827.55/chromedriver' # Use the discovered path
    service = webdriver.ChromeService(executable_path=CHROMEDRIVER_PATH)
    driver = webdriver.Chrome(service=service, options=options)

    try:
        driver.get(show_url)

        # Wait for the description element to be present
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div#about.showpage__section.container"))
        )

        # Get the page source after JavaScript execution
        selenium_soup = BeautifulSoup(driver.page_source, "html.parser")

        description_div = selenium_soup.select_one("div#about.showpage__section.container")

        description_text = []
        if description_div:
            paragraphs = description_div.select("p")
            for p_tag in paragraphs:
                text = p_tag.get_text(strip=True)
                if text:
                    description_text.append(text)

        if description_text:
            full_description = "\n\n".join(description_text)
            print(f"    DEBUG: Extracted description with Selenium (first 100 chars): {full_description[:100]}...")
            return full_description
        else:
            print(f"    DEBUG: No description extracted within #about div for {show_url}")
            return ""

    except TimeoutException:
        print(f"    Timed out waiting for page or description element on {show_url} with Selenium.")
        return ""
    except Exception as e:
        print(f"    Error fetching {show_url} with Selenium: {e}")
        return ""
    finally:
        driver.quit()
        time.sleep(delay)

def crawl_broadway_shows(delay=1.0):
    print(f"📄 正在讀取 Broadway 列表頁: {BROADWAY_SHOWS_URL}")
    resp = requests.get(
        BROADWAY_SHOWS_URL,
        timeout=20,
        headers={
            "User-Agent": USER_AGENT
        },
    )
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    print("---- HTML Snippet from crawl_broadway_shows (first 1000 chars) ----")
    print(soup.prettify()[:1000])
    print("---- End HTML Snippet from crawl_broadway_shows ----")

    all_listing_shows = []
    # Select the containers for individual show cards based on manual inspection
    # Updated selector based on the debugger output
    show_card_elements = soup.select("a.swiper-carousel-card")
    print(f"DEBUG: Found {len(show_card_elements)} a.swiper-carousel-card elements on listing page.") # Added debug

    for card_container in show_card_elements:
        show_data = parse_broadway_listing_card(card_container)
        if show_data:
            all_listing_shows.append(show_data)
        else:
            print(f"DEBUG: parse_broadway_listing_card returned None for a show card container. Snippet: {card_container.prettify()[:200]}") # Added debug

    print(f"✅ 本次從列表頁爬到 {len(all_listing_shows)} 筆 Broadway 節目基本資訊")

    final_shows = []
    for show in all_listing_shows:
        # Using the updated fetch_broadway_detail_description that now uses Selenium
        description = fetch_broadway_detail_description(show["url"])
        show["description"] = description
        final_shows.append(show)

    df = pd.DataFrame(final_shows, columns=BROADWAY_HEADER)
    print(f"✅ 本次爬到 {len(df)} 筆 Broadway 節目資訊 (含詳細描述)")
    return df

In [ ]:
HAMILTON_DETAIL_URL = "https://www.broadway.com/shows/hamilton-broadway/"
print(f"Fetching detail HTML from: {HAMILTON_DETAIL_URL}")
resp = requests.get(
    HAMILTON_DETAIL_URL,
    timeout=20,
    headers={
        "User-Agent": USER_AGENT
    },
)
resp.raise_for_status()
soup = BeautifulSoup(resp.text, "html.parser")

print("---- HTML Snippet (Targeting description on Hamilton detail page) ----")

# Try to find the description using existing selectors first
description_tag_1 = soup.select_one("div.show-details__about-content p")
description_tag_2 = soup.select_one("div.about-section p")
description_tag_3 = soup.select_one("div.body-copy p")

if description_tag_1:
    print("Found with div.show-details__about-content p:")
    print(description_tag_1.prettify()[:1000])
elif description_tag_2:
    print("Found with div.about-section p:")
    print(description_tag_2.prettify()[:1000])
elif description_tag_3:
    print("Found with div.body-copy p:")
    print(description_tag_3.prettify()[:1000])
else:
    print("Description not found with current selectors. Printing a broader section...")
    # Print a broader section if specific selectors fail
    about_section = soup.select_one("div.show-details__about-content")
    if about_section:
        print("--- div.show-details__about-content (first 2000 chars) ---")
        print(about_section.prettify()[:2000])
    else:
        # Fallback to printing body if about section not found
        print("--- Body content (first 2000 chars) ---")
        print(soup.body.prettify()[:2000])

print("---- End HTML Snippet ----")

### Scraping JavaScript-Rendered Content with Selenium

As suspected, the `requests` library only fetches the initial HTML, which doesn't include the dynamically loaded show descriptions. To address this, we'll use **Selenium**, a powerful tool for automating web browsers. Selenium can open a browser (in headless mode, meaning no visible browser window), execute JavaScript, wait for content to load, and then allow us to parse the fully rendered HTML.

First, we need to install Selenium and the Chrome WebDriver (chromedriver).

In [ ]:
# Install Selenium and chromedriver
!pip install selenium
!apt-get update
!apt install chromium-chromedriver chromium-browser

In [ ]:
# Re-run the installation cell to ensure chromium-browser is installed
!pip install selenium
!apt-get update
!apt install chromium-chromedriver chromium-browser

In [ ]:
# Install google-chrome-stable (more reliable for Selenium in Colab)
!apt-get update
# Add Google Chrome's official APT repository
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo "deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main" >> /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install -y google-chrome-stable

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup # Import BeautifulSoup

# Setup Selenium with headless Chrome
options = webdriver.ChromeOptions()
options.add_argument('--headless')       # Run in headless mode
options.add_argument('--no-sandbox')     # Bypass OS security model
options.add_argument('--disable-dev-shm-usage') # Overcome limited resource problems
options.add_argument('--disable-gpu') # Added for stability in headless environments
options.add_argument('--window-size=1920,1080') # Added to specify window size
options.add_argument('--disable-extensions') # Added to disable extensions
options.add_argument('--disable-setuid-sandbox') # Added for more comprehensive sandbox disabling

# Explicitly specify the Chrome binary location for google-chrome-stable
options.binary_location = '/usr/bin/google-chrome'

# Point to the chromedriver executable compatible with a newer Chrome version
CHROMEDRIVER_PATH = '/root/.cache/selenium/chromedriver/linux64/149.0.7827.55/chromedriver' # Path for newer chromedriver
service = webdriver.ChromeService(executable_path=CHROMEDRIVER_PATH)
driver = webdriver.Chrome(service=service, options=options)

print(f"Fetching detail HTML from: {HAMILTON_DETAIL_URL} using Selenium")
driver.get(HAMILTON_DETAIL_URL)

try:
    # Wait for a key element of the description to be present
    # Based on the user's suggestion, we'll look for a div with id 'about' and class 'showpage__section container'
    # Then we'll try to find paragraphs within it.
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "div#about.showpage__section.container"))
    )
    print("Page loaded successfully with description section.")

    # Get the page source after JavaScript execution
    selenium_soup = BeautifulSoup(driver.page_source, "html.parser")

    print("---- HTML Snippet from Selenium (Targeting description on Hamilton detail page) ----")
    description_div = selenium_soup.select_one("div#about.showpage__section.container")

    if description_div:
        paragraphs = description_div.select("p")
        description_text = []
        for p_tag in paragraphs:
            text = p_tag.get_text(strip=True)
            if text:
                description_text.append(text)

        if description_text:
            full_description_selenium = "\n\n".join(description_text)
            print(f"Found description with Selenium (first 500 chars):\n{full_description_selenium[:500]}...")
        else:
            print("Found #about div, but no paragraphs within it. Showing div content:")
            print(description_div.prettify()[:1000])
    else:
        print("Description section 'div#about.showpage__section.container' not found even with Selenium. Showing body:")
        print(selenium_soup.body.prettify()[:1000])

    print("---- End HTML Snippet from Selenium ----")

except TimeoutException:
    print("Timed out waiting for page to load or element to be present with Selenium.")
    print("--- Body content from Selenium (first 2000 chars) ---")
    print(driver.page_source[:2000])
    print("---- End HTML Snippet from Selenium ----")
finally:
    driver.quit()

## 4. 執行爬蟲並寫入 Google Sheet

這一格會：

1. 從 Google Sheet 讀取既有資料
2. 爬取新的 Broadway.com 資料
3. 合併並用 `post_id` 去重
4. 寫回 Google Sheet
5. 再讀一次確認真的寫入成功


In [ ]:
new_shows_df = crawl_broadway_shows(delay=1.0)

old_shows_df = read_sheet_df(ws_broadway, BROADWAY_HEADER)
print(f"📌 Google Sheet 原本有 {len(old_shows_df)} 筆")

broadway_shows_df = pd.concat([old_shows_df, new_shows_df], ignore_index=True)
broadway_shows_df = broadway_shows_df.drop_duplicates(subset=["show_id"], keep="last")
broadway_shows_df = broadway_shows_df.sort_values(by="fetched_at", ascending=False)

written_count = write_sheet_df(ws_broadway, broadway_shows_df, BROADWAY_HEADER)
print(f"✅ 已寫入 Google Sheet：{written_count} 筆")

verify_df = read_sheet_df(ws_broadway, BROADWAY_HEADER)
print(f"🔍 從 Google Sheet 重新讀回：{len(verify_df)} 筆")

if len(verify_df) == written_count:
    print("✅ 寫入驗證成功")
else:
    print("⚠️ 寫入筆數與讀回筆數不同，請檢查 Google Sheet 權限或資料格式")

## 5. 從 Google Sheet 建立 RAG 索引

重點：RAG 不直接吃剛爬下來的記憶體資料，而是**從 Google Sheet 重新讀回**，這樣才能確認流程真的是：

`PTT → Google Sheet → RAG`


In [ ]:
# Re-read from Google Sheet and apply description filtering
# to ensure RAG index is built only with meaningful content
time.sleep(1) # Add a small delay for Google Sheet API consistency
rag_source_df = read_sheet_df(ws_broadway, BROADWAY_HEADER)
rag_source_df = rag_source_df[rag_source_df["description"].astype(str).str.strip() != ""].copy()
print(f"📚 可用於 RAG 的節目數：{len(rag_source_df)}")
rag_source_df.head()

In [ ]:
print("正在載入多語言 Embedding 模型...")
embedding_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
print("✅ Embedding 模型載入完成")

def build_rag_documents(df):
    docs = []
    for _, row in df.iterrows():
        title = str(row.get("title", ""))
        description = str(row.get("description", ""))
        url = str(row.get("url", ""))

        text = (f"標題：{title}\n" f"描述：{description}")
        docs.append({
            "show_id": str(row.get("show_id", "")),
            "title": title,
            "url": url,
            "text": text,
        })
    return docs


def build_faiss_index(docs):
    if not docs:
        raise ValueError("沒有可建立索引的文件")

    texts = [d["text"] for d in docs]
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    embeddings = embeddings.astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, embeddings


rag_documents = build_rag_documents(rag_source_df)
rag_index, rag_embeddings = build_faiss_index(rag_documents)

print(f"✅ RAG 索引建立完成：{len(rag_documents)} 筆節目資訊，向量維度 {rag_embeddings.shape[1]}")

## 6. Gemini 設定與 RAG 問答

請先在 Colab Secrets 裡建立 `gemini`，內容是你的 Gemini API key。


In [ ]:
api_key = userdata.get("gemini")
if not api_key:
    raise ValueError("找不到 Colab Secret：gemini。請先在 Colab Secrets 新增 Gemini API key。")

genai.configure(api_key=api_key)

# 若你的帳號不支援這個模型，可改成你可用的 Gemini model name
GEMINI_MODEL_NAME = "gemini-3-flash-preview"
llm = genai.GenerativeModel(GEMINI_MODEL_NAME)
print(f"✅ Gemini 已設定：{GEMINI_MODEL_NAME}")


In [ ]:
def retrieve_docs(query, k=3):
    if rag_index is None or not rag_documents:
        return []
    q_emb = embedding_model.encode([query], convert_to_numpy=True).astype("float32")
    distances, indices = rag_index.search(q_emb, int(k))

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        if idx == -1:
            continue
        doc = rag_documents[idx].copy()
        doc["distance"] = float(dist)
        results.append(doc)
    return results


def query_rag(question, k=3):
    docs = retrieve_docs(question, k=k)
    if not docs:
        return "找不到相關百老匯節目資料。"

    context = "\n\n---\n\n".join(
        [f"來源標題：{d['title']}\n來源網址：{d['url']}\n{d['text']}" for d in docs]
    )

    prompt = f"""
你是一個根據百老匯節目資料回答問題的助教。
請只根據【百老匯節目資料】回答。
如果資料不足，請明確說「目前資料不足，無法判斷」，不要自行編造。
回答請使用繁體中文，並在最後列出參考來源標題與網址。
回答內容請保持簡潔，限制在 200 字以內。

【百老匯節目資料】
{context}

【問題】
{question}

【回答】
""".strip()

    # 增加 timeout 參數並限制 max_output_tokens
    response = llm.generate_content(prompt, request_options={"timeout": 1200}, generation_config=genai.GenerationConfig(max_output_tokens=200))
    return response.text

## 7. 快速測試


In [ ]:
question = input("請輸入問題：")
answer = query_rag(question, k=3)
print(answer)


## 常見錯誤檢查

如果 PTT 資料沒有寫回 Google Sheet，請依序檢查：

1. 是否有成功印出 `已開啟試算表`，且名稱正確。
2. `SHEET_URL` 是否是你要寫入的那一份 Google Sheet。
3. Google Sheet 權限是否允許目前 Colab 登入的 Google 帳號編輯。
4. 是否執行到「執行爬蟲並寫入 Google Sheet」那一格。
5. 是否有看到 `寫入驗證成功`。
6. RAG 要從 `rag_source_df = read_sheet_df(...)` 開始，確保資料來源是 Google Sheet，而不是記憶體中的暫存變數。


In [ ]:
# 安裝 Gradio
!pip install -q gradio

In [ ]:
import gradio as gr

def gradio_query_rag(question_input):
    """Wrapper function for query_rag to be used with Gradio."""
    if not question_input or not question_input.strip():
        return "請輸入您的問題。"
    print(f"DEBUG: 收到問題，開始檢索文件：{question_input[:50]}...")
    # query_rag 內部會調用 retrieve_docs 和 llm.generate_content
    answer = query_rag(question_input, k=3)
    print("DEBUG: Gemini 回答生成完畢。")
    return answer


# 創建 Gradio 介面
iface = gr.Interface(
    fn=gradio_query_rag,
    inputs=gr.Textbox(lines=2, placeholder="請輸入關於百老匯節目的問題...", label="您的問題"),
    outputs=gr.Textbox(label="答案"),
    title="百老匯節目 RAG 問答機器人",
    description="請輸入您想了解的百老匯節目資訊，我會從資料中為您尋找答案。"
)

# 啟動 Gradio 介面
print("啟動 Gradio 介面中...")
iface.launch(debug=False, share=False) # share=True 可以生成一個公開連結，但有時效性